In [16]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated,List
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver
import operator

from dotenv import load_dotenv
load_dotenv()
model=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [17]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    topic:str
    joke:str
    explanation:str


In [18]:
def generate_joke(state:ChatState)->ChatState:
    print(state)
    response = state['topic']
    prompt=f"Generate a joke about {response}"
    result=model.invoke(prompt).content
    return {"joke":result}

def generate_explanation(state:ChatState)->ChatState:
    response = state['joke']
    prompt=f"Generate an explanation for the joke {response}"
    result=model.invoke(prompt).content
    return {"explanation":result}

In [30]:
checkpointers = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explanation",generate_explanation)


graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","generate_explanation")
graph.add_edge("generate_explanation",END)

workflow=graph.compile(checkpointer=checkpointers)

config={"configurable":{"thread_id":"3"}}
workflow.invoke({"topic":"Shivam"},config=config)

{'topic': 'Shivam'}


{'topic': 'Shivam',
 'joke': 'Since I don\'t know *your* Shivam, these are generic jokes that could apply to anyone named Shivam! Hopefully, one of them gets a chuckle:\n\n1.  Why did Shivam bring a ladder to the bar?\n    ...Because he heard the drinks were on the house!\n\n2.  What\'s Shivam\'s favorite kind of music?\n    ...Heavy metal! Because he\'s a *rock* star! (A playful nod to Lord Shiva\'s powerful imagery, as Shivam means "Lord Shiva")\n\n3.  What did Shivam say when he finally finished his giant project?\n    ..."Phew, that was a *Shivam* of work!"',
 'explanation': 'The humor in these jokes comes primarily from **puns** and **wordplay**, often playing on common idioms or the sound/meaning of the name "Shivam." The preamble is key, as it acknowledges that the jokes are generic attempts since the joke-teller doesn\'t know the specific "Shivam" they\'re addressing.\n\nLet\'s break them down:\n\n1.  **Why did Shivam bring a ladder to the bar? ...Because he heard the drinks we

In [31]:
config1 = {"configurable":{"thread_id":"3"}}
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Shivam', 'joke': 'Since I don\'t know *your* Shivam, these are generic jokes that could apply to anyone named Shivam! Hopefully, one of them gets a chuckle:\n\n1.  Why did Shivam bring a ladder to the bar?\n    ...Because he heard the drinks were on the house!\n\n2.  What\'s Shivam\'s favorite kind of music?\n    ...Heavy metal! Because he\'s a *rock* star! (A playful nod to Lord Shiva\'s powerful imagery, as Shivam means "Lord Shiva")\n\n3.  What did Shivam say when he finally finished his giant project?\n    ..."Phew, that was a *Shivam* of work!"', 'explanation': 'The humor in these jokes comes primarily from **puns** and **wordplay**, often playing on common idioms or the sound/meaning of the name "Shivam." The preamble is key, as it acknowledges that the jokes are generic attempts since the joke-teller doesn\'t know the specific "Shivam" they\'re addressing.\n\nLet\'s break them down:\n\n1.  **Why did Shivam bring a ladder to the bar? ...Because he 